# 03 · Trade cost against quality with routing modes

*Does Cost mode lower spend without dropping below the policy gate — and does Quality mode earn its premium?*

The router ships three modes — **Cost**, **Balanced**, **Quality** — that shift how
aggressively it reaches for stronger models. Same workload, three named router
deployments; we measure whether Cost holds the policy-accuracy gate while lowering
spend, and whether Quality's lift justifies the extra tokens.

> **Skeleton.** This rung is scaffolded: the header, scoring toolkit, and scorecard
> are ready, and the lever cell has a working starting point to refine as you test.

```mermaid
flowchart LR
    Cost[model-router-cost] --> S[Benchmark]
    Bal[model-router balanced] --> S
    Qual[model-router-quality] --> S
    S --> Sc[Scorecard per mode]
    Sc --> D{Best mode holds policy gate<br/>at acceptable cost?}
```

## 1. Install dependencies and load config

In [ ]:
## 1. Ensure dependencies and read .env
import importlib.util, subprocess, sys

_needed = {
    "azure-ai-projects": "azure.ai.projects", "azure-identity": "azure.identity",
    "openai": "openai", "python-dotenv": "dotenv", "pandas": "pandas", "matplotlib": "matplotlib",
}
_missing = [pkg for pkg, mod in _needed.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *_missing])
else:
    print("All dependencies present.")

from dotenv import load_dotenv
load_dotenv(".env")
print(".env loaded from this folder (if present).")

## 2. Authenticate and open the project

In [ ]:
## 2. Project client
import os
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

endpoint = os.environ["MICROSOFT_FOUNDRY_ENDPOINT"]
credential = DefaultAzureCredential()
project = AIProjectClient(endpoint=endpoint, credential=credential)
print("Project client ready.")

## 3. Scoring toolkit

Reuses the shared [Contoso benchmark](../../../demos/contoso-travel/benchmark/) so the
workload stays fixed across every rung.

In [ ]:
## 3. Scoring toolkit (shared shape with notebooks 01–02)
import sys, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../../../demos/contoso-travel/benchmark").resolve()))
from run_benchmark import load_json, grade  # noqa: E402

BENCH = Path("../../../demos/contoso-travel/benchmark")
QUERIES = load_json(BENCH / "queries.json")["queries"]

def run_suite(ask, label, queries=QUERIES, limit=None) -> pd.DataFrame:
    rows = []
    for q in (queries[:limit] if limit else queries):
        turns = q.get("turns") or [q["query"]]
        start = time.perf_counter()
        try:
            text, model, tin, tout = ask(turns)
        except Exception as exc:
            text, model, tin, tout = f"ERROR: {exc}", "", 0, 0
        passed, _ = grade(q, text)
        rows.append({"run": label, "id": q["id"], "category": q["category"], "gate": q["gate"],
                     "model": model or "-", "passed": passed,
                     "latency_ms": round((time.perf_counter() - start) * 1000),
                     "tokens_in": tin, "tokens_out": tout})
    return pd.DataFrame(rows)

def make_model_caller(deployment: str):
    """Call a fixed deployment (or a named router deployment) directly."""
    client = project.get_openai_client()
    def ask(turns):
        conv = client.conversations.create().id
        text = model = ""; tin = tout = 0
        for turn in turns:
            resp = client.responses.create(conversation=conv, input=turn, model=deployment)
            text = resp.output_text
            model = getattr(resp, "model", "") or deployment
            u = getattr(resp, "usage", None)
            if u:
                tin += getattr(u, "input_tokens", 0) or 0
                tout += getattr(u, "output_tokens", 0) or 0
        return text, model, tin, tout
    return ask

def compare(frames, order=None):
    allruns = pd.concat(frames, ignore_index=True)
    summary = allruns.groupby("run").agg(
        quality_pct=("passed", lambda s: round(100 * s.mean(), 1)),
        policy_pct=("gate", lambda s: None),
        avg_tokens=("tokens_out", "mean"),
        p50_ms=("latency_ms", "median"),
    )
    # policy accuracy computed separately (gate == 'policy')
    pol = allruns[allruns["gate"] == "policy"].groupby("run")["passed"].mean().mul(100).round(1)
    summary["policy_pct"] = pol
    if order:
        summary = summary.reindex([o for o in order if o in summary.index])
    return allruns, summary

## 4. Change the lever

Deploy two extra routers in the portal — one in **Cost** mode, one in **Quality**
mode — and add their names to `.env` as `AZURE_MODEL_ROUTER_COST_DEPLOYMENT` and
`AZURE_MODEL_ROUTER_QUALITY_DEPLOYMENT`. Balanced is your existing `model-router`.

In [ ]:
## 4. Define the three routing modes
configs = {"cost": os.getenv("AZURE_MODEL_ROUTER_COST_DEPLOYMENT"),
           "balanced": os.getenv("AZURE_MODEL_ROUTER_DEPLOYMENT"),
           "quality": os.getenv("AZURE_MODEL_ROUTER_QUALITY_DEPLOYMENT")}
configs = {k: v for k, v in configs.items() if v}
print("Modes to compare:", configs)
# frames = [run_suite(make_model_caller(dep), label) for label, dep in configs.items()]

## 5. Score and compare

Run each configuration over the benchmark, then read the four-dimension scorecard —
quality, policy accuracy, cost (tokens), latency — plus the selected-model mix.

In [ ]:
## 5. Run the comparison
# TODO: build `frames` by running run_suite(...) for each configuration above.
# Example once the lever cell defines the callers/deployments:
#   frames = [run_suite(make_model_caller(dep), label) for label, dep in configs.items()]
#   allruns, summary = compare(frames, order=list(configs))
#   display(summary)
frames = []
if frames:
    allruns, summary = compare(frames)
    display(summary)
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    summary["quality_pct"].plot.bar(ax=ax[0], color="#4C78A8", title="Quality %"); ax[0].set_ylim(0, 100)
    summary["avg_tokens"].plot.bar(ax=ax[1], color="#E45756", title="Avg output tokens (cost)")
    plt.tight_layout(); plt.show()
else:
    print("Fill in the lever cell and the frames list above, then re-run.")

## 6. Your Turn

- **Set the gate.** Pick your minimum acceptable policy accuracy. Which modes clear it?
- **Price the premium.** What's the token delta between Balanced and Quality, and is
  the quality gain above the 0.03 noise floor?

## 7. Summary

Routing mode is a single dial from cost to quality. We measured all three against a
fixed policy gate and picked the cheapest mode that still clears it. Next, **notebook
04** constrains the *model subset* to cut cost and routing variance further.

## 8. References

- [Model router concepts — routing modes](https://learn.microsoft.com/en-us/azure/foundry/openai/concepts/model-router)
- [Evaluate model router for your workload](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/evaluate-model-router)
- [Capsule overview](README.md) · [Glossary](../../../docs/GLOSSARY.md)